<a href="https://colab.research.google.com/github/Somaskandan931/flyrank-ml-2026-Somaskandan931/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Somaskandan931/flyrank-ml-2026-Somaskandan931/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub pandas scikit-learn

from huggingface_hub import login
from google.colab import userdata
import duckdb, pandas as pd

login(token=userdata.get('HF_TOKEN'))

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET (TYPE huggingface, TOKEN '" + userdata.get('HF_TOKEN') + "')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{BASE}/fact_content_daily_performance/**/*.parquet"

## 1. Unit of analysis + time window

One row in `fact_content_daily_performance` represents one **(content_hash_id, client_hash_id,
report_date)** combination — one piece of content's daily GSC/GA4 performance for one client.

Decision-moment design for this contract: within March 2026, I split the month into
- an **early window**, `report_date BETWEEN '2026-03-01' AND '2026-03-15'` (source of features), and
- a **late window**, `report_date BETWEEN '2026-03-16' AND '2026-03-31'` (source of the label),

so the label is a genuinely future outcome relative to the features, entirely inside the
same development month. The final month (June 2026 / the `_sample` file) is never touched —
it is the sealed test window, per the warehouse card.

This is an unbalanced panel: `client_has_gsc` / `client_has_ga4` and
`gsc_data_available` / `ga4_data_available` vary per row, so coverage is not uniform
across clients or the full month.

## 2. Fields: feature / label / context / excluded

**Features** (aggregated over the early window, `2026-03-01`–`2026-03-15`)
- `gsc_impressions_early` — sum of `gsc_impressions`, known before the decision cutoff
- `gsc_clicks_early` — sum of `gsc_clicks`, known before the decision cutoff
- `gsc_avg_position_early` — impression-weighted position from `gsc_sum_position`/`gsc_impressions`, early window only
- `ga4_sessions_early` — sum of `ga4_sessions`, known before the decision cutoff
- `scroll_events_early` — sum of `scroll_events`, known before the decision cutoff

**Label**
- `gsc_clicks_late` — sum of `gsc_clicks` over the late window (`2026-03-16`–`2026-03-31`) — the future outcome being predicted

**Context**
- `content_hash_id`, `client_hash_id` — identify the observation, not predictive

**Excluded**
- `gsc_avg_position` (raw, whole-month) — excluded because computing it over the full month would blend in late-window (post-decision) data, causing leakage
- `ga4_total_engagement_sec` from the late window — excluded because it's measured in the same period as the label and would leak the outcome

## 3. Verify it with queries

Three checks on `fact_content_daily_performance` for March 2026: grain, row count / date span, availability.

In [ ]:
q_grain = f"""
SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS n
FROM read_parquet('{FACT_DAILY}')
WHERE month = '2026-03'
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 10
"""
con.sql(q_grain).df()  # empty result confirms one row per (content, client, date)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,report_date,n


In [ ]:
q_span = f"""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('{FACT_DAILY}')
WHERE month = '2026-03'
"""
con.sql(q_span).df()

,n_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
q_avail = f"""
SELECT COUNT(*) AS total_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet('{FACT_DAILY}')
WHERE month = '2026-03'
"""
con.sql(q_avail).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


## 4. Data limits

This slice is a single unbalanced panel month: `client_has_gsc` / `client_has_ga4` and
`gsc_data_available` / `ga4_data_available` show that not every client-content-day has both
GSC and GA4 coverage, so GA4-based features (sessions, engagement, scroll events) will be
directionally weaker or missing for GSC-only rows. Splitting one month into a 15/16-day
early/late window also means the "future" window is short — real seasonal or multi-week
trend effects can't be separated from noise at this scale.

## 5. Five features + the leakage trap

Feature frame built by aggregating `fact_content_daily_performance` per `(content_hash_id,
client_hash_id)` over the early window, joined to the late-window label, filtered to
available rows only.

In [ ]:
df = con.sql(f"""
WITH early AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions_early,
           SUM(gsc_clicks)      AS gsc_clicks_early,
           SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position_early,
           SUM(ga4_sessions)    AS ga4_sessions_early,
           SUM(scroll_events)   AS scroll_events_early
    FROM read_parquet('{FACT_DAILY}')
    WHERE month = '2026-03'
      AND report_date BETWEEN '2026-03-01' AND '2026-03-15'
      AND gsc_data_available IS TRUE
    GROUP BY 1,2
),
late AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_clicks) AS gsc_clicks_late
    FROM read_parquet('{FACT_DAILY}')
    WHERE month = '2026-03'
      AND report_date BETWEEN '2026-03-16' AND '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY 1,2
)
SELECT e.*, l.gsc_clicks_late
FROM early e
JOIN late l USING (content_hash_id, client_hash_id)
""").df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,gsc_impressions_early,gsc_clicks_early,gsc_avg_position_early,ga4_sessions_early,scroll_events_early,gsc_clicks_late
0,content_05597932fe4da067,client_73cda7b4e4f265ea,18.0,0.0,4.833333,NaN,NaN,0.0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,4173.0,6.0,6.265037,NaN,NaN,1.0
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,89.0,0.0,3.471910,NaN,NaN,0.0
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,245.0,0.0,4.085714,NaN,NaN,0.0
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,3705.0,3.0,6.297706,NaN,NaN,3.0


- `gsc_impressions_early` — knowable at the decision moment: summed only over days `03-01`–`03-15`, strictly before the label window.
- `gsc_clicks_early` — same: early-window aggregate only.
- `gsc_avg_position_early` — computed from early-window `gsc_sum_position`/`gsc_impressions` only, no late-window data.
- `ga4_sessions_early` — same: early-window aggregate only.
- `scroll_events_early` — same: early-window aggregate only.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np

feature_cols = ['gsc_impressions_early','gsc_clicks_early',
                'gsc_avg_position_early','ga4_sessions_early','scroll_events_early']

df = df.dropna(subset=feature_cols + ['gsc_clicks_late'])

# Binary label: did clicks in the late window exceed the median? (simple, honest proxy)
median_clicks = df['gsc_clicks_late'].median()
df['label_high_clicks_late'] = (df['gsc_clicks_late'] > median_clicks).astype(int)

X = df[feature_cols]
y = df['label_high_clicks_late']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print("Honest AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))

# --- Deliberate leak: built directly from the label's source column ---
df['leak_column'] = df['gsc_clicks_late'] * 0.9 + df['gsc_clicks_early']

X_leaky = df[feature_cols + ['leak_column']]
X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=0)
model_leaky = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print("Leaky AUC (artificially inflated):", roc_auc_score(y_test, model_leaky.predict_proba(X_test)[:, 1]))

# Remove the leak — keep the honest number
del df['leak_column']

Honest AUC: 0.8623873731551457
Leaky AUC (artificially inflated): 1.0


Adding `leak_column`, derived directly from `gsc_clicks_late`, pushed AUC from 0.86 to a
perfect 1.0 — an artificial, dishonest result, since a real model can never see the future
outcome it's trying to predict. Removing the leaked column restores the honest AUC of 0.86,
which is the number that should actually be reported.

## Self-check
- [x] Every section filled — markdown thinking AND the code that backs it
- [x] Notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit repo URL